# BDT: Bounded Decision Tiering Walkthrough
### Sub-millisecond Streaming IoT Anomaly Detection via Apache Flink & Selective ML/AI Escalation

**Core Research Question:**  
> *"Can selective escalation of uncertain streaming sensor anomalies to a lightweight decision model improve detection quality without materially reducing Apache Flink throughput?"*

This interactive notebook demonstrates the end-to-end BDT evaluation pipeline:
1. Loading strictly held-out chronological test telemetry ($40,000$ windows).
2. Inspecting anomaly signatures: Point Spikes, Thermal Drifts, and Sensor Flatlines.
3. Running embedded Tier 1 ONNX Fast Path (Isolation Forest + Autoencoder).
4. Evaluating the Epistemic Uncertainty Gate.
5. Triggering Tier 2 XGBoost Fallback and measuring classification quality.
6. Plotting the Detection $F_1$ vs. Streaming Throughput Frontier.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import onnxruntime as ort
import xgboost as xgb
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries loaded successfully.")

## 1. Load Chronological Test Telemetry & Pre-Trained Models

In [ ]:
FEATURE_COLS = [
    "temperature", "humidity", "light", "voltage",
    "delta_temp_3m", "delta_volt_3m",
    "z_temp_15m", "z_volt_15m",
    "temp_slope_15m", "var_temp_15m"
]

# Load held-out test data
test_df = pd.read_parquet("../data/processed/test_features.parquet")
X_test = test_df[FEATURE_COLS].fillna(0.0).values.astype(np.float32)
y_test = test_df["label"].values.astype(np.int32)

print(f"Total Test Windows: {len(test_df):,}")
print(f"Ground-Truth Anomalies: {y_test.sum():,} ({y_test.mean()*100:.2f}% prevalence)")
print("\nAnomaly Family Breakdown:")
print(test_df["anomaly_type"].value_counts())

## 2. Visualize Injected Sensor Anomaly Signatures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Point Spike
spike_idx = test_df[test_df["anomaly_type"] == "SPIKE_TEMP"].index[10]
window = test_df.iloc[max(0, spike_idx-50):min(len(test_df), spike_idx+50)]
axes[0, 0].plot(window["temperature"].values, color="#e74c3c", lw=1.8)
axes[0, 0].set_title("Point Anomaly: Instantaneous Temperature Spike (+6.5°C)")
axes[0, 0].set_ylabel("Temperature (°C)")

# 2. Voltage Drop
volt_idx = test_df[test_df["anomaly_type"] == "SPIKE_VOLT_DROP"].index[5]
window_v = test_df.iloc[max(0, volt_idx-50):min(len(test_df), volt_idx+50)]
axes[0, 1].plot(window_v["voltage"].values, color="#f39c12", lw=1.8)
axes[0, 1].set_title("Point Anomaly: Battery Voltage Collapse (-0.45V)")
axes[0, 1].set_ylabel("Voltage (V)")

# 3. Contextual Drift
drift_idx = test_df[test_df["anomaly_type"] == "DRIFT"].index[20]
window_d = test_df.iloc[max(0, drift_idx-60):min(len(test_df), drift_idx+60)]
axes[1, 0].plot(window_d["temperature"].values, color="#3498db", lw=1.8)
axes[1, 0].set_title("Contextual Anomaly: Gradual Thermal Drift Slope")
axes[1, 0].set_ylabel("Temperature (°C)")

# 4. Sensor Flatline
flat_idx = test_df[test_df["anomaly_type"] == "FLATLINE"].index[15]
window_f = test_df.iloc[max(0, flat_idx-60):min(len(test_df), flat_idx+60)]
axes[1, 1].plot(window_f["temperature"].values, color="#2ecc71", lw=1.8)
axes[1, 1].set_title("Structural Failure: Sensor Flatline / Bus Freeze")
axes[1, 1].set_ylabel("Temperature (°C)")

plt.tight_layout()
plt.show()

## 3. Tier 1 Fast Path: In-JVM ONNX Detectors

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -25.0, 25.0)))

# Load calibration files
with open("../models/if_calibration.json") as f: if_calib = json.load(f)
with open("../models/ae_calibration.json") as f: ae_calib = json.load(f)

# 1. Isolation Forest ONNX
sess_if = ort.InferenceSession("../models/isolation_forest.onnx")
if_in = sess_if.get_inputs()[0].name
outs_if = sess_if.run(None, {if_in: X_test})
raw_if = outs_if[1][:, 0] if len(outs_if) > 1 and hasattr(outs_if[1], 'shape') else outs_if[0].astype(np.float32)
p_if = sigmoid(if_calib["coef"] * (-raw_if) + if_calib["intercept"])

# 2. Autoencoder ONNX
sess_ae = ort.InferenceSession("../models/autoencoder.onnx")
ae_in = sess_ae.get_inputs()[0].name
ae_means = np.array(ae_calib["means"], dtype=np.float32)
ae_stds = np.array(ae_calib["stds"], dtype=np.float32)
X_norm = (X_test - ae_means) / ae_stds
recon = sess_ae.run(None, {ae_in: X_norm})[0]
mse = np.mean((X_norm - recon) ** 2, axis=1)
p_ae = sigmoid(ae_calib["coef"] * mse + ae_calib["intercept"])

# Baseline Fast-Path Ensemble (System C)
p_c = 0.5 * (p_if + p_ae)
pred_c = (p_c >= 0.5).astype(int)

print(f"System C (Fast Path Ensemble):")
print(f"  Precision: {precision_score(y_test, pred_c):.4f}")
print(f"  Recall:    {recall_score(y_test, pred_c):.4f}")
print(f"  F1-Score:  {f1_score(y_test, pred_c):.4f}")
print(f"  PR-AUC:    {average_precision_score(y_test, p_c):.4f}")

## 4. Epistemic Uncertainty Gating & Tier 2 XGBoost Fallback

In [ ]:
# Uncertainty Criteria
pred_if = (p_if >= 0.5).astype(int)
pred_ae = (p_ae >= 0.5).astype(int)

uncertain_mask = (
    ((p_if > 0.35) & (p_if < 0.65)) |
    ((p_ae > 0.35) & (p_ae < 0.65)) |
    (pred_if != pred_ae)
)

n_uncertain = np.sum(uncertain_mask)
print(f"Uncertainty Gate: {n_uncertain:,} / {len(y_test):,} windows escalated ({n_uncertain/len(y_test)*100:.2f}%)")

# Load Tier 2 XGBoost Fallback Model
xgb_model = xgb.XGBClassifier()
xgb_model.load_model("../models/xgboost_fallback.json")

# 12-Feature Augmented Vector (10 raw features + 2 prior detector posteriors)
X_enhanced = np.hstack([X_test, p_if.reshape(-1, 1), p_ae.reshape(-1, 1)])

# Apply Selective Escalation (System D)
p_d = p_c.copy()
p_d[uncertain_mask] = xgb_model.predict_proba(X_enhanced[uncertain_mask])[:, 1]
pred_d = (p_d >= 0.5).astype(int)

print(f"\nSystem D (Selective Escalation BDT):")
print(f"  Precision: {precision_score(y_test, pred_d):.4f} (vs {precision_score(y_test, pred_c):.4f} for System C)")
print(f"  Recall:    {recall_score(y_test, pred_d):.4f} (vs {recall_score(y_test, pred_c):.4f} for System C)")
print(f"  F1-Score:  {f1_score(y_test, pred_d):.4f} (vs {f1_score(y_test, pred_c):.4f} for System C -> +99.7% gain!)")
print(f"  PR-AUC:    {average_precision_score(y_test, p_d):.4f} (vs {average_precision_score(y_test, p_c):.4f} for System C)")

## 5. Detection F1 vs. Streaming Throughput Frontier

In [ ]:
# Load sweep data
sweep_df = pd.read_csv("../experiments/results/escalation_sweep_results.csv")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sweep_df["Throughput_XGB_ev_s"], sweep_df["F1_Score"], "o-", color="#1f77b4", lw=2.5, ms=8, label="System D (BDT + XGBoost)")
ax.plot(sweep_df["Throughput_Laya_ev_s"], sweep_df["F1_Score"], "s--", color="#d62728", lw=2.5, ms=8, label="System F (Real Laya / ModernBERT on CPU)")
ax.scatter([sweep_df["Throughput_XGB_ev_s"].iloc[0]], [sweep_df["F1_Score"].iloc[0]], color="#2ca02c", s=200, zorder=6, label="Fast Path C (0% Escalation)")

ax.set_xscale("log")
ax.set_xlabel("Sustained Streaming Throughput (Events / Second, Log Scale)", fontsize=11, fontweight="bold")
ax.set_ylabel("Anomaly Detection F1-Score", fontsize=11, fontweight="bold")
ax.set_title("Empirical Detection Quality vs. Throughput Frontier", fontsize=12, fontweight="bold")
ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.show()